# 2026-07-28 atlas + time since fire (CyVerse)

Reads **MESMA outputs** from:

`new_data/2026-07-28/mesma/<tile>/`

Example (what you already have):

`new_data/2026-07-28/mesma/11_S_KR_2026_7_28/11_S_KR_2026_7_28_mesma_..._fractions`

Upload these **four files** into the same VICE folder as this notebook (overwrite old copies):
- `cyverse_pipeline_2026_07_28.ipynb` (this notebook)
- `export_time_since_fire.py`
- `build_atlas_cyverse.py`
- `postfire_landscape_atlas1.html`

Your VICE folder already has the four scripts next to `new_data/`. The atlas reads:
- MESMA fractions: `new_data/2026-07-28/mesma/`
- Time since fire: `new_data/Time_since_fire/`

**How to run (Kernel → Restart, then in order):**
1. Run the inventory code cell. Confirm it prints those two paths.
2. Run the time-since-fire cell (Earth Engine login if asked).
3. Run the build-atlas cell. Outputs go to `new_data/2026-07-28/atlas/`.

This notebook is set to convert **5 images first** (`MAX_TILES = 5`). After the build, run the **clickable tile map** cell so `atlas/index.html` shows Sentinel-2 squares you can click. Set `MAX_TILES = 0` later for all tiles.

In [1]:
from pathlib import Path

DATE = '2026-07-28'
# Which scene(s): 'all', one folder ('11_S_KR'), or comma-separated names
TILE = 'all'
# Trial run: only the first N matching cubes. Set to 0 for every tile.
MAX_TILES = 5
cwd = Path.cwd().resolve()

def find_mesma_dir():
    checks = []
    for parent in [cwd, *cwd.parents]:
        checks.append(parent / 'new_data' / DATE / 'mesma')
        checks.append(parent / DATE / 'mesma')
        if parent.name == 'mesma' and parent.parent.name == DATE:
            checks.append(parent)
    checks.extend([
        Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data') / DATE / 'mesma',
        Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data') / DATE / 'mesma',
        Path('/home/jovyan/data-store/SCE-CU/sentinel2_data/new_data') / DATE / 'mesma',
        Path('/data-store/SCE-CU/sentinel2_data/new_data') / DATE / 'mesma',
    ])
    for p in checks:
        if p.exists():
            return p
    raise FileNotFoundError('Could not find new_data/2026-07-28/mesma')

MESMA = find_mesma_dir()
DATE_DIR = MESMA.parent
NEW = DATE_DIR.parent
TSF = NEW / 'Time_since_fire'

print('MESMA folder:', MESMA)
print()
tile_dirs = sorted(p for p in MESMA.iterdir() if p.is_dir())
print(f'Tile folders: {len(tile_dirs)}')
fracs = []
for tile in tile_dirs:
    cubes = [p for p in tile.iterdir() if p.is_file() and p.name.endswith('_fractions') and p.suffix == '']
    print(f'  {tile.name}')
    for c in cubes:
        print(f'      {c.name}  ({c.stat().st_size/1e9:.1f} GB)')
        fracs.append(c)
selected = fracs
if TILE and str(TILE).lower() != 'all':
    needles = [t.strip().replace('-', '_').upper() for t in str(TILE).split(',') if t.strip()]
    selected = [c for c in fracs if any(n in c.parent.name.upper() or n in c.name.upper() for n in needles)]
if MAX_TILES:
    selected = selected[: int(MAX_TILES)]
print(f'\nFractions cubes found: {len(fracs)}')
print('TILE setting:', TILE, ' MAX_TILES:', MAX_TILES)
print(f'Will convert {len(selected)} cube(s) for the HTML:')
for c in selected:
    print(f'  {c.parent.name}')
print('TSF folder:', TSF, 'exists' if TSF.exists() else 'MISSING')

MESMA folder: /home/jovyan/data-store/new_data/2026-07-28/mesma

Tile folders: 42
  10_S_GB_2026_7_28
      10_S_GB_2026_7_28_mesma_20260830T203227_fractions  (6.8 GB)
  11_S_KR_2026_7_28
      11_S_KR_2026_7_28_mesma_20260830T211316_fractions  (6.8 GB)
  11_S_KS_2026_7_28
      11_S_KS_2026_7_28_mesma_20260831T013621_fractions  (6.8 GB)
  11_S_KT_2026_7_28
      11_S_KT_2026_7_28_mesma_20260831T045048_fractions  (6.8 GB)
  11_S_KU_2026_7_28
      11_S_KU_2026_7_28_mesma_20260831T065633_fractions  (6.8 GB)
  11_S_KV_2026_7_28
      11_S_KV_2026_7_28_mesma_20260831T075029_fractions  (6.8 GB)
  11_S_LA_2026_7_28
      11_S_LA_2026_7_28_mesma_20260831T080145_fractions  (6.8 GB)
  11_S_LB_2026_7_28
      11_S_LB_2026_7_28_mesma_20260831T104710_fractions  (6.8 GB)
  11_S_LR_2026_7_28
      11_S_LR_2026_7_28_mesma_20260831T124321_fractions  (6.8 GB)
  11_S_LS_2026_7_28
      11_S_LS_2026_7_28_mesma_20260831T172729_fractions  (6.8 GB)
  11_S_LT_2026_7_28
      11_S_LT_2026_7_28_mesma_20260831

## 1. Export time-since-fire GeoTIFF

Writes MODIS days-since-burn through **2026-07-28** into `new_data/Time_since_fire/`.
Authenticate Earth Engine if prompted (project `timesincefire`).

In [2]:
# from IPython import get_ipython

# TSF.mkdir(parents=True, exist_ok=True)
# ts_out = TSF / f'days_since_last_burn_{DATE.replace("-", "")}.tif'
# cmd = f'export_time_since_fire.py --end-date {DATE} --out {ts_out}'
# print('Running', cmd)
# get_ipython().run_line_magic('run', cmd)

# print('TSF files in', TSF)
# if TSF.exists():
#     for p in sorted(TSF.rglob('*')):
#         if p.is_file():
#             print(f'  {p.name:50s} {p.stat().st_size/1e6:8.1f} MB')
# else:
#     print('Time_since_fire folder not found yet')

Running export_time_since_fire.py --end-date 2026-07-28 --out /home/jovyan/data-store/new_data/Time_since_fire/days_since_last_burn_20260728.tif
Earth Engine project: timesincefire
Building TSF through 2026-07-28 (MODIS MCD64A1) ...
Writing /home/jovyan/data-store/new_data/Time_since_fire/days_since_last_burn_20260728.tif
Exporting with geemap.ee_export_image ...
Generating URL ...
Please wait ...
Data downloaded to /home/jovyan/data-store/new_data/Time_since_fire/days_since_last_burn_20260728.tif
Done: /home/jovyan/data-store/new_data/Time_since_fire/days_since_last_burn_20260728.tif (0.1 MB)
TSF files in /home/jovyan/data-store/new_data/Time_since_fire
  days_since_last_burn_20260728.tif                       0.1 MB


## 2. Build the atlas

Uses `TILE` and `MAX_TILES` from cell 1 (trial: first 5 cubes). Writes small files to:

`new_data/2026-07-28/atlas/`

- `index.html` — pick a scene
- `<tile>/` — HTML + JS + PNG for that image

Download those small files — not the GB cubes. In the atlas UI, use **Scene to analyse** to switch tiles.

In [10]:
from IPython import get_ipython

html = Path.cwd() / 'postfire_landscape_atlas1.html'
if not html.exists():
    html = Path('/home/jovyan/data-store/postfire_landscape_atlas1.html')
to_build = []
for frac in selected:
    out = DATE_DIR / 'atlas' / frac.parent.name
    if (out / 'mesma_grid.js').exists():
        print('skip', frac.parent.name)
        continue
    to_build.append(frac)
print(f'Building {len(to_build)} remaining of {len(selected)} selected')

for frac in to_build:
    out = DATE_DIR / 'atlas' / frac.parent.name
    cmd = (
        f'build_atlas_cyverse.py --fractions {frac} --tsf-dir {TSF} '
        f'--html {html} --out {out}'
    )
    print('\n====', frac.parent.name, '====')
    try:
        get_ipython().run_line_magic('run', cmd)
    except SystemExit as exc:
        print('skipped:', frac.parent.name, exc)

atlas = DATE_DIR / 'atlas'
print('\nAtlas outputs:')
if atlas.exists():
    for p in sorted(atlas.rglob('*')):
        if p.is_file():
            print(f'  {p.relative_to(atlas)}  {p.stat().st_size/1e6:.2f} MB')
else:
    print('No atlas folder yet')

NameError: name 'selected' is not defined

## 2b. Clickable tile map

Rewrites `atlas/index.html` into a map of Sentinel-2 squares. Click a square to open that scene. Does not rebuild MESMA. Run after the build cell (or after inventory if tiles are already built).

In [ ]:
#!/usr/bin/env python3
"""Rewrite atlas/index.html into a clickable Sentinel-2 tile map.

Does not rebuild MESMA. Run on CyVerse after tiles exist:

  python write_tile_index.py
  python write_tile_index.py /home/jovyan/data-store/new_data/2026-07-28/atlas
"""

from __future__ import annotations

import json
import math
import re
import sys
from pathlib import Path

BAND_SOUTH = {
    "C": -80, "D": -72, "E": -64, "F": -56, "G": -48, "H": -40, "J": -32, "K": -24,
    "L": -16, "M": -8, "N": 0, "P": 8, "Q": 16, "R": 24, "S": 32, "T": 40, "U": 48,
    "V": 56, "W": 64, "X": 72,
}


def tile_label(folder: str) -> tuple[str, str]:
    name = Path(folder).name
    tile, date = name, "2026-07-28"
    if "_S_" in name:
        parts = name.split("_")
        try:
            tile = f"{parts[0]}{parts[1]}{parts[2]}"
            date = f"{parts[3]}-{int(parts[4]):02d}-{int(parts[5]):02d}"
        except Exception:
            pass
    return tile, date


def utm_to_latlon(easting: float, northing: float, zone: int) -> tuple[float, float]:
    a, f, k0 = 6378137.0, 1 / 298.257223563, 0.9996
    e = math.sqrt(2 * f - f * f)
    e1 = (1 - math.sqrt(1 - e * e)) / (1 + math.sqrt(1 - e * e))
    x = easting - 500000.0
    lon0 = math.radians((zone - 1) * 6 - 180 + 3)
    m = northing / k0
    mu = m / (a * (1 - e**2 / 4 - 3 * e**4 / 64 - 5 * e**6 / 256))
    phi1 = (
        mu
        + (3 * e1 / 2 - 27 * e1**3 / 32) * math.sin(2 * mu)
        + (21 * e1**2 / 16 - 55 * e1**4 / 32) * math.sin(4 * mu)
        + (151 * e1**3 / 96) * math.sin(6 * mu)
    )
    sinp, cosp, tanp = math.sin(phi1), math.cos(phi1), math.tan(phi1)
    ep2 = e * e / (1 - e * e)
    n1 = a / math.sqrt(1 - e * e * sinp * sinp)
    t1, c1 = tanp * tanp, ep2 * cosp * cosp
    r1 = a * (1 - e * e) / ((1 - e * e * sinp * sinp) ** 1.5)
    d = x / (n1 * k0)
    lat = phi1 - (n1 * tanp / r1) * (
        d * d / 2
        - (5 + 3 * t1 + 10 * c1 - 4 * c1 * c1 - 9 * ep2) * d**4 / 24
        + (61 + 90 * t1 + 298 * c1 + 45 * t1 * t1 - 252 * ep2 - 3 * c1 * c1) * d**6 / 720
    )
    lon = lon0 + (
        d
        - (1 + 2 * t1 + c1) * d**3 / 6
        + (5 - 2 * c1 + 28 * t1 - 3 * c1 * c1 + 8 * ep2 + 24 * t1 * t1) * d**5 / 120
    ) / cosp
    return math.degrees(lon), math.degrees(lat)


def s2_tile_ring(label: str):
    raw = re.sub(r"[^0-9A-Z]", "", label.upper())
    m = re.match(r"(\d{1,2})([C-HJ-NP-X])([A-HJ-NP-Z])([A-HJ-NP-V])", raw)
    if not m:
        return None
    zone = int(m.group(1))
    band, col, row = m.group(2), m.group(3), m.group(4)
    cols = "ABCDEFGHJKLMNPQRSTUVWXYZ"
    set_off = ((zone - 1) % 3) * 8
    rel = (cols.index(col) - set_off) % 24
    easting = (rel % 8 + 1) * 100000
    rows = "ABCDEFGHJKLMNPQRSTUV" if zone % 2 else "FGHJKLMNPQRSTUVABCDE"
    row_i = rows.index(row)
    lat0 = BAND_SOUTH[band]
    a, f = 6378137.0, 1 / 298.257223563
    e2 = 2 * f - f * f
    phi = math.radians(lat0)
    mer = a * (
        (1 - e2 / 4 - 3 * e2**2 / 64 - 5 * e2**3 / 256) * phi
        - (3 * e2 / 8 + 3 * e2**2 / 32 + 45 * e2**3 / 1024) * math.sin(2 * phi)
        + (15 * e2**2 / 256 + 45 * e2**3 / 1024) * math.sin(4 * phi)
        - (35 * e2**3 / 3072) * math.sin(6 * phi)
    )
    n_band = mer * 0.9996
    northing = row_i * 100000 + round((n_band - row_i * 100000) / 2_000_000) * 2_000_000
    while northing + 100000 < n_band:
        northing += 2_000_000
    while northing > n_band + 2_000_000:
        northing -= 2_000_000
    ring = []
    for x, y in (
        (easting, northing),
        (easting + 100000, northing),
        (easting + 100000, northing + 100000),
        (easting, northing + 100000),
    ):
        lon, lat = utm_to_latlon(x, y, zone)
        ring.append([round(lat, 5), round(lon, 5)])
    return ring


def ring_from_grid(js_path: Path):
    try:
        raw = js_path.read_text(encoding="utf-8", errors="replace").split("=", 1)[1].strip()
        if raw.endswith(";"):
            raw = raw[:-1]
        ll = json.loads(raw).get("lonlat") or {}
        w, e, s, n = ll.get("west"), ll.get("east"), ll.get("south"), ll.get("north")
        if None in (w, e, s, n):
            return None
        return [[s, w], [s, e], [n, e], [n, w]]
    except Exception:
        return None


INDEX_HTML = """<!DOCTYPE html>
<html lang="en"><head>
<meta charset="UTF-8"/><meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>Postfire recovery atlas — choose a scene</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<style>
  :root{--page:#E9DFC1;--card:#F7F1DD;--ink:#221F17;--ink-soft:#5C5645;}
  body{margin:0;background:var(--page);color:var(--ink);font-family:system-ui,sans-serif;}
  .wrap{max-width:1100px;margin:0 auto;padding:28px 18px 60px;}
  h1{font-size:28px;margin:0 0 8px;}
  p{color:var(--ink-soft);line-height:1.5;max-width:640px;}
  #tileMap{height:min(70vh,720px);border:2px solid var(--ink);border-radius:8px;margin:18px 0;}
  .grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(200px,1fr));gap:12px;}
  .scene{display:flex;flex-direction:column;gap:3px;background:var(--card);border:1.5px solid var(--ink);
    border-radius:10px;padding:14px 16px;text-decoration:none;color:var(--ink);}
  .scene:hover,.scene.active{background:#EFE5C7;border-color:#2F5233;}
  .scene-id{font-weight:700;font-size:18px;}
  .scene-date{font-size:13px;color:var(--ink-soft);}
  .scene-folder{font-size:11px;color:#8C826A;}
</style></head><body><div class="wrap">
<h1>Postfire recovery atlas</h1>
<p>Click a yellow square on the map to open that Sentinel-2 MESMA scene.</p>
<div id="tileMap"></div>
<div class="grid">__CARDS__</div>
</div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="atlas_tiles.js"></script>
<script>
const tiles = window.ATLAS_TILES || [];
const map = L.map('tileMap').setView([34.8, -118.8], 6);
L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}', {
  attribution: 'Tiles &copy; Esri'
}).addTo(map);
const layers = {}, group = [];
function setActive(id){
  document.querySelectorAll('.scene').forEach(a => a.classList.toggle('active', a.dataset.id === id));
  Object.entries(layers).forEach(([k, poly]) => {
    poly.setStyle(k === id
      ? {color:'#2F5233', weight:3, fillColor:'#B9862E', fillOpacity:0.45}
      : {color:'#F0B429', weight:2, fillColor:'#F0B429', fillOpacity:0.18});
  });
}
tiles.forEach(t => {
  if (!t.ring) return;
  const poly = L.polygon(t.ring, {color:'#F0B429', weight:2, fillColor:'#F0B429', fillOpacity:0.18}).addTo(map);
  poly.bindTooltip(t.label, {sticky:true});
  poly.on('mouseover', () => setActive(t.id));
  poly.on('click', () => { window.location.href = './' + t.id + '/postfire_landscape_atlas1.html'; });
  layers[t.id] = poly;
  group.push(poly);
});
if (group.length) map.fitBounds(L.featureGroup(group).getBounds().pad(0.08));
document.querySelectorAll('.scene').forEach(a => a.addEventListener('mouseenter', () => setActive(a.dataset.id)));
</script></body></html>
"""


def find_atlas() -> Path:
    cwd = Path.cwd().resolve()
    checks = [
        cwd / "new_data" / "2026-07-28" / "atlas",
        Path("/home/jovyan/data-store/new_data/2026-07-28/atlas"),
        Path("/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/atlas"),
    ]
    for parent in [cwd, *cwd.parents]:
        checks.append(parent / "new_data" / "2026-07-28" / "atlas")
        if parent.name == "atlas":
            checks.append(parent)
    for p in checks:
        if p.exists() and any((c / "mesma_grid.js").exists() for c in p.iterdir() if c.is_dir()):
            return p
    raise FileNotFoundError("Could not find atlas/ with mesma_grid.js tile folders")


def write_tile_index(atlas: Path) -> None:
    folders = [
        p.name
        for p in sorted(atlas.iterdir())
        if p.is_dir()
        and p.name != "test"
        and (p / "mesma_grid.js").exists()
        and (p / "postfire_landscape_atlas1.html").exists()
    ]
    entries = []
    for folder in folders:
        label, date = tile_label(folder)
        ring = s2_tile_ring(label) or ring_from_grid(atlas / folder / "mesma_grid.js")
        entries.append({"id": folder, "label": label, "date": date, "ring": ring})
    js = "window.ATLAS_TILES=" + json.dumps(entries, separators=(",", ":")) + ";\n"
    (atlas / "atlas_tiles.js").write_text(js)
    for folder in folders:
        (atlas / folder / "atlas_tiles.js").write_text(js)
    cards = "\n".join(
        f'<a class="scene" data-id="{e["id"]}" href="./{e["id"]}/postfire_landscape_atlas1.html">'
        f'<span class="scene-id">{e["label"]}</span>'
        f'<span class="scene-date">{e["date"]}</span>'
        f'<span class="scene-folder">{e["id"]}</span></a>'
        for e in entries
    )
    dest = atlas / "index.html"
    dest.write_text(INDEX_HTML.replace("__CARDS__", cards or "<p>No atlases built yet.</p>"))
    n_rings = sum(1 for e in entries if e["ring"])
    print(f"Wrote clickable map: {dest}")
    print(f"tiles: {len(entries)}   squares on map: {n_rings}")
    if "tileMap" not in dest.read_text():
        raise RuntimeError("index.html write failed — still the old card page")

atlas = DATE_DIR / 'atlas'
print('Writing map into', atlas)
write_tile_index(atlas)
text = (atlas / 'index.html').read_text()
print('index.html has tileMap:', 'tileMap' in text)
print('index.html first line of body title check:', 'yellow square' in text or 'Click a Sentinel-2 tile' in text)
print('DONE. Next: run the Open cell, then Cmd+Shift+R. Do NOT run the Build cell after this.')


## Open the HTML on CyVerse (do not download the GB cubes)

The `atlas/` folder is only ~20 MB. Jupyter's built-in HTML preview will look empty — it blocks `mesma_grid.js`.

Run the next cell for clickable links, **or** in the file browser: open a tile folder → right-click `postfire_landscape_atlas1.html` → **Open in New Browser Tab**.

In [19]:
from pathlib import Path
import shutil

html = Path('/home/jovyan/data-store/postfire_landscape_atlas1.html')
atlas = DATE_DIR / 'atlas' if 'DATE_DIR' in dir() else None
if atlas is None or not atlas.exists():
    for cand in (
        Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/atlas'),
        Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/atlas'),
        Path('/home/jovyan/data-store/new_data/2026-07-28/atlas'),
    ):
        if cand.exists():
            atlas = cand
            break
print('atlas:', atlas)
if atlas is None or not Path(atlas).exists():
    raise FileNotFoundError('atlas/ not found. Run the inventory cell first.')
tsf = Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire/southern_california_burn_map.html')
if not tsf.exists():
    tsf = Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire/southern_california_burn_map.html')

shutil.copy2(tsf, atlas / 'southern_california_burn_map.html')
print('burn map:', atlas / 'southern_california_burn_map.html', 'exists', (atlas / 'southern_california_burn_map.html').exists())

for d in sorted(atlas.iterdir()):
    if d.is_dir() and (d / 'mesma_grid.js').exists():
        (d / 'postfire_landscape_atlas1.html').write_text(html.read_text())
        print('updated', d.name)

burn map: /home/jovyan/data-store/new_data/2026-07-28/atlas/southern_california_burn_map.html exists True
updated 10_S_GB_2026_7_28
updated 11_S_KR_2026_7_28
updated 11_S_KS_2026_7_28
updated 11_S_KT_2026_7_28
updated 11_S_KU_2026_7_28


In [11]:
import os
from IPython.display import display, HTML

atlas = DATE_DIR / 'atlas'
size_mb = sum(p.stat().st_size for p in atlas.rglob('*') if p.is_file()) / 1e6
print(f'atlas/ size: {size_mb:.1f} MB  (this is what the HTML needs — not the GB MESMA cubes)')

prefix = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
if not prefix.endswith('/'):
    prefix += '/'

roots = []
for r in (Path.cwd(), Path('/home/jovyan/data-store'), Path('/home/jovyan')):
    try:
        roots.append((r, atlas.resolve().relative_to(r.resolve())))
    except ValueError:
        pass
rel = roots[0][1] if roots else Path(atlas.name)
base = f'{prefix}files/{rel.as_posix()}'

rows = [f'<p><b>Open in a new browser tab</b> (not Jupyter preview):</p>']
rows.append(f'<p><a href="{base}/index.html" target="_blank">Scene picker (index.html)</a></p><ul>')
for folder in sorted(p for p in atlas.iterdir() if p.is_dir()):
    html_name = folder / 'postfire_landscape_atlas1.html'
    if html_name.exists():
        rows.append(f'<li><a href="{base}/{folder.name}/postfire_landscape_atlas1.html" target="_blank">{folder.name}</a></li>')
rows.append('</ul>')
rows.append('<p>If a link 404s, in the file browser go into the tile folder, right-click the HTML → Open in New Browser Tab.</p>')
display(HTML('\n'.join(rows)))

NameError: name 'DATE_DIR' is not defined

In [ ]:
from pathlib import Path
import os, socket, subprocess, time
from IPython.display import display, HTML

atlas = DATE_DIR / 'atlas'
write_tile_index(atlas)
print('index.html has tileMap:', 'tileMap' in (atlas / 'index.html').read_text())

PORT = 8765

def port_open(port):
    with socket.socket() as s:
        s.settimeout(0.3)
        return s.connect_ex(('127.0.0.1', port)) == 0

if not port_open(PORT):
    subprocess.Popen(
        ['python3', '-m', 'http.server', str(PORT), '--bind', '127.0.0.1'],
        cwd=str(atlas), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    time.sleep(0.8)

prefix = os.environ.get('JUPYTERHUB_SERVICE_PREFIX') or '/'
if not prefix.endswith('/'):
    prefix += '/'
url = f'{prefix}proxy/{PORT}/index.html?v=map3'
print(url)
display(HTML(
    f'<p style="font-size:18px"><a href="{url}" target="_blank" rel="noopener">'
    f'<b>Open tile map</b></a></p>'
    f'<p>You must see a street map with yellow squares. If you still see only cards, this is a cached tab — close it and click this link again.</p>'
))


In [21]:
from pathlib import Path

needles = [
    Path.cwd() / 'postfire_landscape_atlas1.html',
    Path('/home/jovyan/data-store/postfire_landscape_atlas1.html'),
    Path('/home/jovyan/postfire_landscape_atlas1.html'),
]
src = next((p for p in needles if p.exists() and 'zoomFitBtn' in p.read_text()), None)
print('source:', src)
if src is None:
    raise SystemExit('New HTML not found. Upload postfire_landscape_atlas1.html into the same folder as the notebook.')

atlas = DATE_DIR / 'atlas' if 'DATE_DIR' in dir() else None
if atlas is None or not atlas.exists():
    for cand in (
        Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/atlas'),
        Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/atlas'),
        Path('/home/jovyan/data-store/new_data/2026-07-28/atlas'),
    ):
        if cand.exists():
            atlas = cand
            break
print('atlas:', atlas)
if atlas is None or not Path(atlas).exists():
    raise FileNotFoundError('atlas/ not found. Run the inventory cell first.')
text = src.read_text()
for d in sorted(atlas.iterdir()):
    dest = d / 'postfire_landscape_atlas1.html'
    if d.is_dir() and dest.exists():
        dest.write_text(text)
        print(d.name, 'now has Fit:', 'zoomFitBtn' in dest.read_text())

source: /home/jovyan/data-store/postfire_landscape_atlas1.html
10_S_GB_2026_7_28 now has Fit: True
11_S_KR_2026_7_28 now has Fit: True
11_S_KS_2026_7_28 now has Fit: True
11_S_KT_2026_7_28 now has Fit: True
11_S_KU_2026_7_28 now has Fit: True
